# WN18RR Quick Validation

**Goal:** Verify GP-KGE OOD detection advantage generalizes to WN18RR

**Settings (reduced for speed):**
- Epochs: 30 (vs 50)
- Embedding dim: 100 (vs 200)
- MRR sample: 2000 (vs full)
- Single seed (42)

**Expected runtime:** ~10-15 min on CPU, ~3-5 min on GPU

In [1]:
# Setup
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

Cloning into '/content/kg-bayesian-prior'...
remote: Enumerating objects: 226, done.
remote: Counting objects: 100% (226/226), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 226 (delta 131), reused 160 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (226/226), 192.80 KiB | 1.53 MiB/s, done.
Resolving deltas: 100% (131/131), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.6/280.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.3/176.3 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 5.0 MB/s eta 0:00:00


In [2]:
import gc, json, warnings, time
import torch
import torch.nn.functional as F
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import eigsh
from tqdm.notebook import tqdm

from src.data.loaders import load_wn18rr
from src.models import DistMult, GPKGE
from src.models.ggpn import GGPN
from src.kernels.matern_graph import GraphLaplacian
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cpu


In [3]:
# Load WN18RR
print("Loading WN18RR...")
train_data, _, test_data = load_wn18rr()
print(f"Entities: {train_data.num_entities:,}")
print(f"Relations: {train_data.num_relations}")
print(f"Train: {len(train_data):,}, Test: {len(test_data):,}")

Loading WN18RR...
WN18RR not found. Downloading...


train.txt: 0.00B [00:00, ?B/s]


HTTPError: HTTP Error 404: Not Found

In [ ]:
# Reduced hyperparameters for quick validation
CONFIG = {
    'embedding_dim': 100,
    'epochs': 30,
    'batch_size': 1024,
    'lr': 0.001,
    'mrr_sample': 2000,
    'ece_sample': 2000,
    'ood_sample': 2000,
}
print("Config:", CONFIG)

In [ ]:
def train_model(model, name, train_data):
    """Train a model with BCE loss"""
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])

    for ep in (pbar := tqdm(range(CONFIG['epochs']), desc=name)):
        model.train()
        loss_sum, n = 0, 0
        indices = np.random.permutation(len(train_data))

        for st in range(0, len(indices), CONFIG['batch_size']):
            batch_idx = indices[st:st+CONFIG['batch_size']]
            pos = torch.tensor(train_data.triples[batch_idx], device=device)
            neg = pos.clone()
            neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)

            opt.zero_grad()

            # Get scores
            if hasattr(model, 'score_triple'):
                pos_s = model.score_triple(pos[:,0], pos[:,1], pos[:,2])
                neg_s = model.score_triple(neg[:,0], neg[:,1], neg[:,2])
            else:
                pos_s = model(pos[:,0], pos[:,1], pos[:,2])
                neg_s = model(neg[:,0], neg[:,1], neg[:,2])

            loss = F.binary_cross_entropy_with_logits(
                torch.cat([pos_s, neg_s]),
                torch.cat([torch.ones_like(pos_s), torch.zeros_like(neg_s)]))

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            loss_sum += loss.item()
            n += 1

        pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

    return model

In [ ]:
def evaluate_model(model, name, train_data, test_data):
    """Evaluate MRR, ECE, AUROC"""
    model.eval()
    results = {}

    # MRR (sampled)
    sample_idx = np.random.choice(len(test_data), min(CONFIG['mrr_sample'], len(test_data)), replace=False)
    sample = test_data.triples[sample_idx]

    ranks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sample), 100), desc=f"{name} MRR", leave=False):
            batch = sample[i:i+100]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]

            if hasattr(model, 'score_tails'):
                scores = model.score_tails(h, r)
            else:
                # Fallback for models without score_tails
                all_t = torch.arange(train_data.num_entities, device=device)
                scores = []
                for j in range(len(h)):
                    if hasattr(model, 'score_triple'):
                        s = model.score_triple(h[j].expand(len(all_t)), r[j].expand(len(all_t)), all_t)
                    else:
                        s = model(h[j].expand(len(all_t)), r[j].expand(len(all_t)), all_t)
                    scores.append(s)
                scores = torch.stack(scores)

            target = scores[torch.arange(len(t), device=device), t]
            ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

    ranks = torch.tensor(ranks, dtype=torch.float)
    results['mrr'] = (1/ranks).mean().item()
    results['hits@1'] = (ranks <= 1).float().mean().item()
    results['hits@10'] = (ranks <= 10).float().mean().item()

    # ECE (sampled)
    ece_idx = np.random.choice(len(test_data), min(CONFIG['ece_sample'], len(test_data)), replace=False)
    pos = test_data.triples[ece_idx]
    neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
    all_t = np.vstack([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

    confs = []
    with torch.no_grad():
        for i in range(0, len(all_t), 1024):
            batch = all_t[i:i+1024]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            if hasattr(model, 'score_triple'):
                scores = model.score_triple(h, r, t)
            else:
                scores = model(h, r, t)
            confs.append(torch.sigmoid(scores).cpu().numpy())
    conf = np.concatenate(confs)
    results['ece'], _ = expected_calibration_error(conf, labels)
    results['brier'] = brier_score(conf, labels)

    # AUROC
    id_idx = np.random.choice(len(test_data), min(CONFIG['ood_sample'], len(test_data)), replace=False)
    id_triples = test_data.triples[id_idx]
    ood_triples = create_ood_dataset(train_data, test_data, "random", CONFIG['ood_sample'])

    def get_uncertainty(triples):
        uncs = []
        with torch.no_grad():
            for i in range(0, len(triples), 1024):
                batch = triples[i:i+1024]
                h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]

                if hasattr(model, 'predict_with_uncertainty'):
                    pred = model.predict_with_uncertainty(h, r, t)
                    if isinstance(pred, dict):
                        unc = pred.get('total', pred.get('epistemic', torch.zeros(len(h))))
                    else:
                        unc = pred[1]
                    uncs.append(unc.cpu().numpy())
                else:
                    # Entropy-based uncertainty for baselines
                    if hasattr(model, 'score_triple'):
                        s = model.score_triple(h, r, t)
                    else:
                        s = model(h, r, t)
                    p = torch.sigmoid(s)
                    unc = -p * torch.log(p + 1e-10) - (1-p) * torch.log(1-p + 1e-10)
                    uncs.append(unc.cpu().numpy())
        return np.concatenate(uncs)

    results['auroc'] = compute_auroc(get_uncertainty(id_triples), get_uncertainty(ood_triples))

    return results

In [ ]:
def setup_gpkge_eigendecomp(model, train_data):
    """Setup eigendecomposition for GP-KGE relation-aware kernel"""
    kernel = model.kernel
    kernel.num_entities = train_data.num_entities
    kernel.relation_laplacians = {}

    success, failed = 0, 0
    for rel_id, adj in tqdm(train_data.relation_adjacencies.items(), desc="Eigendecomp", leave=False):
        if adj.nnz < 10:
            continue
        try:
            degrees = np.array(adj.sum(axis=1)).flatten()
            D_inv_sqrt = sparse.diags(1.0 / np.sqrt(np.maximum(degrees, 1e-10)))
            L = sparse.diags(degrees) - adj
            L_norm = D_inv_sqrt @ L @ D_inv_sqrt
            L_norm = (L_norm + L_norm.T) / 2
            k = min(50, L_norm.shape[0] - 2)  # Reduced for speed
            if k < 2:
                continue
            eigvals, eigvecs = eigsh(L_norm, k=k, which='SM', maxiter=500, tol=1e-3)
            kernel.relation_laplacians[rel_id] = GraphLaplacian(adj.shape[0])
            kernel.relation_laplacians[rel_id].eigenvalues = torch.tensor(eigvals, dtype=torch.float32)
            kernel.relation_laplacians[rel_id].eigenvectors = torch.tensor(eigvecs, dtype=torch.float32)
            success += 1
        except:
            failed += 1
    print(f"Eigendecomp: {success} success, {failed} failed (WN18RR has {train_data.num_relations} relations)")

In [ ]:
# Run experiments
set_seed(42)
all_results = {}

print("="*70)
print("WN18RR QUICK VALIDATION")
print("="*70)

In [ ]:
# 1. DistMult (Baseline)
print("\n" + "="*50)
print("Model: DistMult")
print("="*50)

gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

start = time.time()
model = DistMult(train_data.num_entities, train_data.num_relations, CONFIG['embedding_dim'])
model = train_model(model, "DistMult", train_data)
results = evaluate_model(model, "DistMult", train_data, test_data)
results['time'] = time.time() - start

print(f"MRR={results['mrr']:.4f}, H@10={results['hits@10']:.4f}, ECE={results['ece']:.4f}, AUROC={results['auroc']:.4f}")
all_results['DistMult'] = results
del model

In [ ]:
# 2. GGPN
print("\n" + "="*50)
print("Model: GGPN")
print("="*50)

gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

start = time.time()
model = GGPN(
    train_data.num_entities,
    train_data.num_relations * 2,  # Forward + backward
    embedding_dim=CONFIG['embedding_dim'],
    hidden_dim=CONFIG['embedding_dim'],
    num_layers=2,
    num_rff=CONFIG['embedding_dim'],
)
model.set_graph(train_data)
model = train_model(model, "GGPN", train_data)
results = evaluate_model(model, "GGPN", train_data, test_data)
results['time'] = time.time() - start

print(f"MRR={results['mrr']:.4f}, H@10={results['hits@10']:.4f}, ECE={results['ece']:.4f}, AUROC={results['auroc']:.4f}")
all_results['GGPN'] = results
del model

In [ ]:
# 3. GP-KGE (Ours)
print("\n" + "="*50)
print("Model: GP-KGE (Ours)")
print("="*50)

gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

start = time.time()
model = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=CONFIG['embedding_dim'],
    kernel_type="relation_aware",
    num_inducing=min(300, train_data.num_entities),
)
setup_gpkge_eigendecomp(model, train_data)
model = train_model(model, "GP-KGE", train_data)
results = evaluate_model(model, "GP-KGE", train_data, test_data)
results['time'] = time.time() - start

print(f"MRR={results['mrr']:.4f}, H@10={results['hits@10']:.4f}, ECE={results['ece']:.4f}, AUROC={results['auroc']:.4f}")
all_results['GP-KGE'] = results
del model

In [ ]:
# Summary
print("\n" + "="*80)
print("WN18RR RESULTS SUMMARY")
print("="*80)

print(f"\n{'Model':<15} {'MRR':>8} {'H@1':>8} {'H@10':>8} {'ECE↓':>8} {'AUROC↑':>8} {'Time':>8}")
print("-"*75)
for name, r in all_results.items():
    print(f"{name:<15} {r['mrr']:>8.4f} {r['hits@1']:>8.4f} {r['hits@10']:>8.4f} {r['ece']:>8.4f} {r['auroc']:>8.4f} {r['time']:>7.1f}s")

print("\n" + "="*80)
print("KEY QUESTION: Does GP-KGE AUROC advantage generalize?")
print("="*80)

gpkge_auroc = all_results['GP-KGE']['auroc']
ggpn_auroc = all_results['GGPN']['auroc']
distmult_auroc = all_results['DistMult']['auroc']

print(f"\nGP-KGE AUROC: {gpkge_auroc:.4f}")
print(f"GGPN AUROC:   {ggpn_auroc:.4f}")
print(f"DistMult AUROC: {distmult_auroc:.4f}")

if gpkge_auroc > ggpn_auroc and gpkge_auroc > distmult_auroc:
    print("\n✅ YES! GP-KGE OOD advantage generalizes to WN18RR")
else:
    print("\n⚠️ Mixed results - need full experiment to confirm")

In [ ]:
# Save results
output = {
    "dataset": "WN18RR",
    "config": CONFIG,
    "results": all_results,
    "note": "Quick validation with reduced settings"
}

os.makedirs("results", exist_ok=True)
with open("results/wn18rr_quick.json", 'w') as f:
    json.dump(output, f, indent=2, default=float)
print("Saved to results/wn18rr_quick.json")